In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# MODIS NDVI DATASET - PRODUCTION QUALITY CLEANING
# ============================================================

INPUT_FILE = "brics_modis_ndvi_india.csv"
OUTPUT_FILE = "brics_modis_ndvi_india_clean.csv"

# Create output directory
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("MODIS NDVI DATASET CLEANING")
print("=" * 60)

print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_columns = [
    "country",
    "latitude",
    "longitude",
    "date",
    "modis_date",
    "product",
    "satellite",
    "band",
    "pixel_index",
    "ndvi"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Keep expected columns
df = df[required_columns].copy()

# ------------------------------------------------------------
# 3. Clean text columns
# ------------------------------------------------------------

text_columns = [
    "country",
    "modis_date",
    "product",
    "satellite",
    "band"
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
    )

# Standardize country
df["country"] = df["country"].replace({
    "IND": "India",
    "INDIA": "India",
    "india": "India"
})

# ------------------------------------------------------------
# 4. Convert numeric columns
# ------------------------------------------------------------

numeric_columns = [
    "latitude",
    "longitude",
    "pixel_index",
    "ndvi"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 5. Convert date column
# ------------------------------------------------------------

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. Validate latitude
# ------------------------------------------------------------

invalid_latitude = ~df["latitude"].between(
    -90,
    90
)

df.loc[
    invalid_latitude,
    "latitude"
] = pd.NA

# ------------------------------------------------------------
# 7. Validate longitude
# ------------------------------------------------------------

invalid_longitude = ~df["longitude"].between(
    -180,
    180
)

df.loc[
    invalid_longitude,
    "longitude"
] = pd.NA

# ------------------------------------------------------------
# 8. Validate NDVI
# ------------------------------------------------------------

# NDVI theoretically ranges from -1 to +1.

invalid_ndvi = ~df["ndvi"].between(
    -1,
    1
)

invalid_ndvi_count = invalid_ndvi.sum()

df.loc[
    invalid_ndvi,
    "ndvi"
] = pd.NA

# ------------------------------------------------------------
# 9. Validate pixel index
# ------------------------------------------------------------

invalid_pixel_index = (
    df["pixel_index"] < 0
)

invalid_pixel_count = invalid_pixel_index.sum()

df.loc[
    invalid_pixel_index,
    "pixel_index"
] = pd.NA

# ------------------------------------------------------------
# 10. Validate MODIS product
# ------------------------------------------------------------

unexpected_products = (
    df["product"].dropna().unique()
)

print(
    "\nMODIS products found:",
    unexpected_products
)

# ------------------------------------------------------------
# 11. Remove exact duplicates
# ------------------------------------------------------------

rows_before_duplicates = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

duplicates_removed = (
    rows_before_duplicates - len(df)
)

# ------------------------------------------------------------
# 12. Sort spatial-temporal observations
# ------------------------------------------------------------

df = df.sort_values(
    by=[
        "latitude",
        "longitude",
        "date",
        "pixel_index"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 13. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(
    "Rows before cleaning:",
    rows_before_duplicates
)

print(
    "Rows after cleaning:",
    len(df)
)

print(
    "Duplicate rows removed:",
    duplicates_removed
)

print(
    "Invalid latitude values:",
    invalid_latitude.sum()
)

print(
    "Invalid longitude values:",
    invalid_longitude.sum()
)

print(
    "Invalid NDVI values:",
    invalid_ndvi_count
)

print(
    "Invalid pixel indices:",
    invalid_pixel_count
)

print("\nMissing values:")

print(
    df.isnull().sum()
)

print("\nDate range:")

print(
    df["date"].min(),
    "to",
    df["date"].max()
)

print("\nNDVI range:")

print(
    df["ndvi"].min(),
    "to",
    df["ndvi"].max()
)

print("\nData types:")

print(
    df.dtypes
)

# ------------------------------------------------------------
# 14. Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Cleaned file saved to:",
    OUTPUT_FILE
)

MODIS NDVI DATASET CLEANING
Original shape: (14580, 10)

MODIS products found: <StringArray>
['MOD13Q1']
Length: 1, dtype: string

CLEANING SUMMARY
Rows before cleaning: 14580
Rows after cleaning: 14580
Duplicate rows removed: 0
Invalid latitude values: 0
Invalid longitude values: 0
Invalid NDVI values: 0
Invalid pixel indices: 0

Missing values:
country        0
latitude       0
longitude      0
date           0
modis_date     0
product        0
satellite      0
band           0
pixel_index    0
ndvi           0
dtype: int64

Date range:
2026-01-01 00:00:00 to 2026-05-09 00:00:00

NDVI range:
0.0617 to 0.9822

Data types:
country        string[python]
latitude              float64
longitude             float64
date           datetime64[ns]
modis_date     string[python]
product        string[python]
satellite      string[python]
band           string[python]
pixel_index           float64
ndvi                  float64
dtype: object

CLEANING COMPLETED SUCCESSFULLY
Cleaned file saved to: